In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Local Fourier RM 唯一16×16低带宽局部基 V1

直接 Run all：同一 source/seed、同模型revision、同shared initial/pristine native scheduler，四臂 OFF／PAYLOAD_MULTI／STATE_OLD_MULTI／STATE_LOWBAND_MULTI 各真实50采样步，MULTI25–49。原payload、RM32 chips／45states／4ages／alpha=1/sqrt5568／pilot eta696/cap1不变；cap只缩小，不强制放大。

唯一新载体：四16×16 supports (4:20,8:24),(4:20,40:56),(24:40,8:24),(24:40,40:56)，16网格非DC共轭representative按signed频率平方、原(h,w)lex取前32，沿用原密钥置换/符号域。较大支持与较低频率是一个固定构造，不证明单独原因，底部支持触h40边界。不扫描、不改时域family/phase/惩罚/阈值。

generation与VAE/media分子进程释放内存，保存100个真实writer sidecars及实际norm，但sidecars不入receiver。每臂一次现有VAEdecode/clamp＋原np.rint RGB8；同一保存bytes分别DIRECT、原RAW420、原libx264 CRF18/yuv420p/8fps MP4，再12次VAEencode。保存源raster、YUV、两种RGB24、命令/日志/probe及环境。RAW420/MP4差异可含color转换/metadata，不单独归因H264。

两个公开receiver family OLD8／LOWBAND16各自独立读取全部四臂×三通道×两key：48q、96ABS/DIFF、16704path、272448difference edge、95040absolute local costs；同模板/同raster比较不是独立样本。每channel/key仅一次新重复payload，共24read/48message后评。两个family分别报告，不能择优合并。先封存全部raw与snapshots，再truth/message后评；全部acceptance=false。

EXECUTION_COMPLETE仅工程完整，不等于scientific PASS。该轮检验开题中的媒体稳定局部状态载体；时间相关片段payload、独立盲片段/声明时域编辑、片段—序列归因与必要拒绝尚未完成。quality仅已保存bytes的诊断，无质量阈值。当前Python/Torch优先，实际WanPipeline/AutoencoderKLWan和媒体工具可用性缺失才必要修复，无venv/版本/GPU型号门禁。Published source: `3386cbd3edca22f823f5be69aba79b526b45b839`.


In [ ]:
SOURCE_SHA = '3386cbd3edca22f823f5be69aba79b526b45b839'
FIXED = {'source_cases': 1, 'arms': 4, 'channels': 3, 'reader_families': 2, 'generated': 4, 'generation_steps': 200, 'controlled_steps': 75, 'writer_sidecars': 100, 'underlying_observations': 12, 'normalized': 12, 'keys': 2, 'keyed_projections': 48, 'path_score_records': 96, 'valid_paths': 174, 'catalog_per_record': 3915, 'path_costs': 16704, 'difference_edge_costs': 272448, 'absolute_local_state_costs': 95040, 'path_posthoc': 96, 'difference_edge_posthoc': 48, 'payload_reads': 24, 'payload_evaluations': 48, 'catalog_slots': 375840, 'structural_exclusions': 359136, 'quality_pairs': 18}
from pathlib import Path
import datetime,hashlib,json,os,subprocess,sys,traceback
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=Path('/content/drive/MyDrive/Video-WM/Video-Local-Fourier-RM-Lowband-V1')/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
RUN_OUTPUT=OUTPUT/'fixed_reference'
REPO=Path('/content/SC-SSTW-LOCAL-FOURIER-RM-LOWBAND-'+STAMP)
PYTHON=sys.executable
ARMS=('OFF','PAYLOAD_MULTI','STATE_OLD_MULTI','STATE_LOWBAND_MULTI');KEYS=('CORRECT','WRONG')
CHANNELS=('DIRECT_RGB8','RAW420','MP4');FAMILIES=('OLD8','LOWBAND16');MODES=('ABSOLUTE_CONTROL','ADJACENT_DIFFERENCE')
setup=dict(status='SETUP_STARTED',source_sha=SOURCE_SHA,fixed_denominator=FIXED,
 generation={a:{'status':'NOT_RUN_SETUP'} for a in ARMS},writer_diagnostics={a+'/'+str(i):{'status':'NOT_RUN_SETUP'} for a in ARMS for i in range(25,50)},
 terminal_inputs={a:{'status':'NOT_RUN_SETUP'} for a in ARMS},rasters={a:{'status':'NOT_RUN_SETUP'} for a in ARMS},
 transport={a+'/'+c:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS},normalized={a+'/'+c:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS},
 projections={a+'/'+c+'/'+f+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for f in FAMILIES for k in KEYS},
 mode_reads={a+'/'+c+'/'+f+'/'+k+'/'+m:{'status':'NOT_RUN_SETUP','accepted_payload':False,'state_path_accepted':False} for a in ARMS for c in CHANNELS for f in FAMILIES for k in KEYS for m in MODES},
 path_posthoc={a+'/'+c+'/'+f+'/'+k+'/'+m:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for f in FAMILIES for k in KEYS for m in MODES},
 edge_posthoc={a+'/'+c+'/'+f+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for f in FAMILIES for k in KEYS},
 payload_reads={a+'/'+c+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for k in KEYS},
 payload_evaluations={a+'/'+c+'/'+k+'/'+t:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for k in KEYS for t in ('REGISTERED','WRONG_MESSAGE')},
 quality={c+'/'+a+'_vs_'+b:{'status':'NOT_RUN_SETUP'} for c in CHANNELS for i,a in enumerate(ARMS) for b in ARMS[:i]})
def write_json(path,value):
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(value,indent=2)+'\n');os.replace(tmp,path)
def failed(stage,exc):
    setup.update(status='SETUP_FAILED',stage=stage,error=f'{type(exc).__name__}: {exc}')
    write_json(OUTPUT/'setup_receipt.json',setup)
    write_json(OUTPUT/'setup_failure.json',dict(stage=stage,error=str(exc),traceback=traceback.format_exc()))
def logged(command,stage,cwd=None,check=True):
    try:
        with (OUTPUT/'execution.log').open('a') as log:
            log.write('COMMAND '+repr(command)+'\n');log.flush()
            child=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in child.stdout:print(line,end='',flush=True);log.write(line);log.flush()
            code=child.wait();log.write('EXIT '+str(code)+'\n')
        if check and code:raise subprocess.CalledProcessError(code,command)
        return code
    except Exception as exc:failed(stage,exc);raise
write_json(OUTPUT/'setup_receipt.json',setup)
print('Fixed output:',OUTPUT)
if SOURCE_SHA is None:
    error=RuntimeError('Unpublished draft: bind published source SHA before Run all');failed('SOURCE_BINDING',error);raise error


In [ ]:
try:
    logged(['git','clone','--filter=blob:none','https://github.com/RICHAAARC/SC-SSTW.git',str(REPO)],'SOURCE_CLONE')
    logged(['git','-C',str(REPO),'fetch','origin',SOURCE_SHA],'SOURCE_FETCH')
    logged(['git','-C',str(REPO),'checkout','--detach',SOURCE_SHA],'SOURCE_CHECKOUT')
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
    if actual!=SOURCE_SHA or dirty:raise RuntimeError('source SHA/clean checkout mismatch')
    write_json(OUTPUT/'source_receipt.json',dict(source_sha=actual,clean=True))
    import shutil
    tools={name:shutil.which(name) for name in ('ffmpeg','ffprobe')}
    tool_codes={name:logged([path,'-version'],name.upper()+'_PROBE',check=False) if path else None for name,path in tools.items()}
    if any(code!=0 for code in tool_codes.values()):
        try:
            logged(['apt-get','update'],'MEDIA_TOOL_UPDATE',check=False)
            logged(['apt-get','install','-y','ffmpeg'],'MEDIA_TOOL_REPAIR',check=False)
        except Exception as repair_error:print('Media-tool repair error retained:',repair_error)
    setup['media_tool_probe_returncodes']=tool_codes
    torch_ok=logged([PYTHON,'-c','import torch; print(torch.__version__)'],'TORCH_PROBE',check=False)==0
    if not torch_ok:logged([PYTHON,'-m','pip','install','--upgrade','torch'],'TORCH_REPAIR',check=False)
    vae_probe='from diffusers import AutoencoderKLWan,WanPipeline; print(AutoencoderKLWan.__name__,WanPipeline.__name__)'
    vae_ok=logged([PYTHON,'-c',vae_probe],'VAE_IMPORT_PROBE',check=False)==0
    if not vae_ok:
        logged([PYTHON,'-m','pip','install','diffusers>=0.35.1,<0.40','transformers>=4.45,<5','accelerate>=0.34','safetensors>=0.4.5'],'VAE_DEPENDENCY_REPAIR',check=False)
    dependency_code=logged([PYTHON,'-m','pip','check'],'DEPENDENCY_REPORT',check=False)
    setup['dependency_check_returncode']=dependency_code
    if dependency_code:print('Metadata conflicts retained; actual VAE import/load determines availability.')
    (OUTPUT/'environment_freeze.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
    environment_code='import json,torch; from experiments.wan_state_clock.video_local_fourier_rm_lowband_v1_run import environment; r=environment(); r.update(cuda_available=torch.cuda.is_available()); print(json.dumps(r))'
    env=json.loads(subprocess.check_output([PYTHON,'-c',environment_code],cwd=REPO,text=True))
    write_json(OUTPUT/'environment_receipt.json',env)
    if not env['cuda_available']:print('CUDA unavailable: CPU path selected; much slower.')
    setup.update(status='SETUP_COMPLETE',python_executable=PYTHON,environment_strategy='current_python_separate_generation_media_workers')
    write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('SOURCE_OR_ENVIRONMENT',exc);raise


In [ ]:
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
    if token:os.environ['HF_TOKEN']=token
except Exception:
    pass
finally:
    token=None
command=[PYTHON,'-u','-m','experiments.wan_state_clock.video_local_fourier_rm_lowband_v1_run','--output',str(RUN_OUTPUT)]
try:
    returncode=logged(command,'FIXED_RUN',cwd=REPO,check=False)
    RESULT_PATH=RUN_OUTPUT/'result.json'
    write_json(OUTPUT/'execution_receipt.json',dict(command=command,returncode=returncode,result_path=str(RESULT_PATH),result_exists=RESULT_PATH.is_file()))
    if not RESULT_PATH.is_file():raise RuntimeError('No runner result; setup denominator retained')
    setup.update(status='SUPERSEDED_BY_RESULT',result_path=str(RESULT_PATH));write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('RUNNER',exc);raise


In [ ]:
result=json.loads(RESULT_PATH.read_text())
if result['source_sha']!=SOURCE_SHA or result['fixed_denominator']!=FIXED:raise RuntimeError('result identity mismatch')
print('Status:',result['status'],'Counts:',result['counts'])
print('Calls:',result['calls'],'Workers:',result['workers'])
print('Shared before25:',result.get('before_step25_identity'))
for arm,row in result['generation'].items():print('Writer:',arm,row['status'],(row.get('receipt') or {}).get('control_norm_statistics'),(row.get('receipt') or {}).get('writer_diagnostics'))
for family in FAMILIES:
    print('Independent reader family:',family)
    for name,row in result['path_posthoc'].items():
        if '/'+family+'/' in name:print(name,row['status'],'rank=',row.get('true_path_rank'),'Delta=',row.get('true_path_delta'),'matches=',row.get('canonical_path_matches'))
for name,row in result['payload_evaluations'].items():print('Repeated payload:',name,row['status'],'bit errors=',row.get('bit_errors'),'exact=',row.get('exact_bits'))
print('Saved-RGB quality diagnostics:',result['quality'])
from IPython.display import Video,display
for arm in ARMS:
    video=result['transport'].get(arm+'/MP4',{}).get('events',{}).get('mp4',{})
    path=video.get('path')
    if video.get('status')=='SAVED' and isinstance(path,str) and Path(path).is_file():
        try:print('Saved MP4 preview:',arm);display(Video(str(path),embed=True))
        except Exception as preview_error:print('Preview unavailable; result retained:',arm,preview_error)
print('Failures:',result['failures'])
print('Evidence ceiling:',result['evidence_ceiling'])
print('Retained result:',RESULT_PATH)
